In [1]:
!pip install requests beautifulsoup4 pandas tqdm lxml

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import requests
import json
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm

In [3]:
FACTSHEET_URL = "https://www.who.int/news-room/fact-sheets"

In [4]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(
    FACTSHEET_URL,
    headers=headers
)

print(response.status_code)

200


In [5]:
soup = BeautifulSoup(
    response.text,
    "lxml"
)

print(soup.title.text)


	Fact sheets



In [6]:
links = []

for a in soup.find_all("a", href=True):

    href = a["href"]
    text = a.get_text(strip=True)

    if "/fact-sheets/detail/" in href:

        links.append({
            "disease": text,
            "url": "https://www.who.int" + href
        })

In [7]:
print("Total diseases:", len(links))

links[:10]

Total diseases: 240


[{'disease': 'Abortion',
  'url': 'https://www.who.int/news-room/fact-sheets/detail/abortion'},
 {'disease': 'Abuse of older people',
  'url': 'https://www.who.int/news-room/fact-sheets/detail/abuse-of-older-people'},
 {'disease': 'Adolescent and young adult health',
  'url': 'https://www.who.int/news-room/fact-sheets/detail/adolescents-health-risks-and-solutions'},
 {'disease': 'Adolescent pregnancy',
  'url': 'https://www.who.int/news-room/fact-sheets/detail/adolescent-pregnancy'},
 {'disease': 'Ageing and health',
  'url': 'https://www.who.int/news-room/fact-sheets/detail/ageing-and-health'},
 {'disease': 'Alcohol',
  'url': 'https://www.who.int/news-room/fact-sheets/detail/alcohol'},
 {'disease': 'Ambient (outdoor) air pollution',
  'url': 'https://www.who.int/news-room/fact-sheets/detail/ambient-(outdoor)-air-quality-and-health'},
 {'disease': 'Anaemia',
  'url': 'https://www.who.int/news-room/fact-sheets/detail/anaemia'},
 {'disease': 'Animal bites',
  'url': 'https://www.who.int

In [8]:
df = pd.DataFrame(links)

df.to_csv(
    "who_disease_links.csv",
    index=False
)

df.head()

,disease,url
0,Abortion,https://www.who.int/news-room/fact-sheets/deta...
1,Abuse of older people,https://www.who.int/news-room/fact-sheets/deta...
2,Adolescent and young adult health,https://www.who.int/news-room/fact-sheets/deta...
3,Adolescent pregnancy,https://www.who.int/news-room/fact-sheets/deta...
4,Ageing and health,https://www.who.int/news-room/fact-sheets/deta...


In [9]:
test_url = links[0]["url"]

response = requests.get(
    test_url,
    headers=headers
)

soup = BeautifulSoup(
    response.text,
    "lxml"
)

print(soup.title.text)


	Abortion



In [10]:
for tag in soup.find_all(
    ["h2","h3"]
):
    print(tag.get_text(strip=True))

Key facts
Overview
Scope of the problem
Consequences of inaccessible quality abortion care
Expanding quality abortion care
WHO response
Database
Related health topic



In [11]:
def scrape_fact_sheet(url):
    response = requests.get(
        url,
        headers=headers,
        timeout=20
    )
    soup = BeautifulSoup(response.text, "lxml")
    
    disease_name = soup.find("h1").get_text(strip=True)
    
    data = {
        "disease_name": disease_name,
        "url": url,
        "source": "WHO Fact Sheets",
        "sections": [] # Changed from {} to []
    }
    
    headings = soup.find_all(["h2"])
    
    # Extended skip list
    skip_keywords = [
        "who response", 
        "references", 
        "database",
        "related health topic",
        "further reading",
        "related links"
    ]
    
    for heading in headings:
        section_title = heading.get_text(strip=True)
        
        if section_title.lower() in skip_keywords:
            continue
            
        content = []
        current = heading.find_next_sibling()
        
        while current:
            if current.name == "h2":
                break
                
            # Preserve bullet points explicitly
            if current.name in ['ul', 'ol']:
                for li in current.find_all('li'):
                    text = li.get_text(" ", strip=True)
                    if text:
                        content.append(f"- {text}")
            else:
                text = current.get_text(" ", strip=True)
                if text:
                    content.append(text)
                    
            current = current.find_next_sibling()
            
        if content: # Only append if the section actually has text
            data["sections"].append({
                "section_title": section_title,
                "content": "\n".join(content)
            })
            
    return data

In [12]:
sample = scrape_fact_sheet(
    links[0]["url"]
)

print(
    json.dumps(
        sample,
        indent=2
    )[:3000]
)

{
  "disease_name": "Abortion",
  "url": "https://www.who.int/news-room/fact-sheets/detail/abortion",
  "source": "WHO Fact Sheets",
  "sections": [
    {
      "section_title": "Key facts",
      "content": "- Six out of 10 unintended pregnancies end in induced abortion.\n- Abortion is a common health intervention. It is very safe when carried out using a method recommended by WHO, appropriate to the pregnancy duration and by someone with the necessary skills.\n- However, around 45% of abortions are unsafe.\n- Unsafe abortion is an important preventable cause of maternal deaths and morbidities. It can lead to physical and mental health complications and social and financial burdens for women, communities and health systems.\n- Lack of access to safe, timely, affordable and respectful abortion care is a critical public health and human rights issue."
    },
    {
      "section_title": "Overview",
      "content": "Around 73\u00a0million induced abortions take place worldwide each year

In [13]:
all_diseases = []

for item in tqdm(links):

    try:

        disease_data = scrape_fact_sheet(
            item["url"]
        )

        all_diseases.append(
            disease_data
        )

    except Exception as e:

        print(
            f"Failed: {item['disease']}"
        )

        print(e)

  0%|          | 0/240 [00:00<?, ?it/s]

100%|██████████| 240/240 [06:32<00:00,  1.64s/it]


In [15]:
print(
    "Total Diseases:",
    len(all_diseases)
)

for disease in all_diseases[:3]:
    print(disease["disease_name"])
    
    # Extract just the titles from our list of section objects
    section_titles = [section["section_title"] for section in disease["sections"]]
    print(section_titles)

Total Diseases: 240
Abortion
['Key facts', 'Overview', 'Scope of the problem', 'Consequences of inaccessible quality abortion care', 'Expanding quality abortion care']
Abuse of older people
['Key facts', 'Scope of the problem', 'Consequences', 'Risk factors', 'Prevention']
Adolescent and young adult health
['Key facts', 'Overview', 'Main health issues', 'Rights of adolescents']


In [16]:
# Deduplicate in memory
unique_diseases = []
seen = set()

for disease in all_diseases:
    name = disease["disease_name"]
    if name not in seen:
        unique_diseases.append(disease)
        seen.add(name)

print("Original count:", len(all_diseases))
print("Unique count:", len(unique_diseases))

# Save directly to the final Master JSON
with open("who_fact_sheets_master.json", "w", encoding="utf-8") as f:
    json.dump(
        unique_diseases,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Master JSON saved successfully.")

Original count: 240
Unique count: 239
Master JSON saved successfully.


In [17]:
import os

os.makedirs(
    "raw_data/who",
    exist_ok=True
)

print("Folder created.")

Folder created.


In [18]:
import re

def clean_text(text):
    if not text:
        return ""

    # Remove non-breaking spaces
    text = text.replace('\xa0', ' ')
    # Remove excessive inline spaces/tabs
    text = re.sub(r'[ \t]+', ' ', text)

    # Fix mid-sentence line breaks
    lines = [line.strip() for line in text.split('\n')]
    cleaned_lines = []
    
    for line in lines:
        if not line:
            continue
            
        # If this line starts with a lowercase letter AND the previous line 
        # doesn't end with terminal punctuation, they belong together.
        if (cleaned_lines and 
            re.match(r'^[a-z]', line) and 
            not cleaned_lines[-1].endswith(('.', ':', ';', '-'))):
            
            cleaned_lines[-1] = f"{cleaned_lines[-1]} {line}"
        else:
            cleaned_lines.append(line)

    return "\n\n".join(cleaned_lines)

In [19]:
import re

def clean_filename(name):

    name = name.lower()

    name = re.sub(
        r'[^a-z0-9\s_-]',
        '',
        name
    )

    name = name.replace(
        " ",
        "_"
    )

    return f"{name}.txt"

In [20]:
for disease in unique_diseases:
    filename = clean_filename(disease["disease_name"])
    filepath = os.path.join("raw_data/who", filename)

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"Disease: {disease['disease_name']}\n")
        f.write(f"Source: WHO Fact Sheets\n")
        f.write(f"URL: {disease['url']}\n\n")

        # Updated iteration for the new array structure
        for section in disease["sections"]:
            section_title = section["section_title"]
            cleaned_content = clean_text(section["content"])

            f.write(f"=== {section_title} ===\n\n")
            f.write(cleaned_content)
            f.write("\n\n")

print("All txt files created.")

All txt files created.


In [21]:
import os

files = os.listdir(
    "raw_data/who"
)

print(
    "Total txt files:",
    len(files)
)

Total txt files: 239


In [22]:
with open(
    "raw_data/who/cancer.txt",
    "r",
    encoding="utf-8"
) as f:

    print(
        f.read()[:3000]
    )

Disease: Cancer
Source: WHO Fact Sheets
URL: https://www.who.int/news-room/fact-sheets/detail/cancer

=== The problem ===

Cancer is a leading cause of death worldwide, accounting for nearly 10 million deaths in 2022 (1) . The most common in 2022 (in terms of new cases of cancer) were:

- lung (2.5 million cases)

- breast (2.3 million cases)

- colon and rectum (1.9 million cases)

- prostate (1.5 million cases)

- skin (non-melanoma) (1.2 million cases)

- stomach (1.0 million cases).

The most common causes of cancer death in 2022 were:

- lung (1.82 million deaths)

- colon and rectum (904 000 deaths)

- liver (760 000 deaths)

- breast (666 000 deaths)

- stomach (660 000 deaths).

Each year, approximately 400 000 children develop cancer. The most common cancers vary between countries, and cervical cancer is the most common cancer among women in 21 countries, primarily in sub-Saharan Africa.

=== Causes ===

Cancer arises from the transformation of normal cells into tumour cells i

In [23]:
import json
import re
from datetime import datetime

# 1. Load the current master JSON
with open("who_fact_sheets_master.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

structured_data = []
current_date = datetime.now().strftime("%Y-%m-%d")

# Regex to find WHO citations like (1), (2,3), etc., preceded by a space
citation_pattern = re.compile(r'\s*\(\d+(?:,\d+)*\)\s*')

for item in raw_data:
    disease_name = item["disease_name"]
    
    # Create a safe, unique doc_id
    safe_name = re.sub(r'[^a-z0-9]', '_', disease_name.lower())
    safe_name = re.sub(r'_+', '_', safe_name).strip('_')
    doc_id = f"who_{safe_name}"
    
    # Process and clean sections
    cleaned_sections = []
    for section in item["sections"]:
        # Remove the citation numbers
        clean_text = re.sub(citation_pattern, ' ', section["content"])
        # Clean up any double spaces left behind
        clean_text = re.sub(r' +', ' ', clean_text).strip()
        
        cleaned_sections.append({
            "section_title": section["section_title"],
            "text": clean_text
        })
        
    # Build the structured document
    structured_doc = {
        "doc_id": doc_id,
        "title": disease_name,
        "source_name": item["source"],
        "source_url": item["url"],
        "category": "Disease & Awareness", # Standardized category
        "topic": disease_name,
        "retrieved_on": current_date,
        "content": cleaned_sections
    }
    
    structured_data.append(structured_doc)

# 2. Save the final, unified Master JSON
with open("who_structured_master.json", "w", encoding="utf-8") as f:
    json.dump(structured_data, f, ensure_ascii=False, indent=2)

print(f"Successfully structured {len(structured_data)} documents!")

Successfully structured 239 documents!
